# SmolVLA Memory SFT — Kaggle 2×T4

- **Accelerator**: Chọn **GPU T4 ×2** (Dual GPU)
- **Internet**: Chuyển sang **On** (để cài packages và tải checkpoint/tokenizer)
- **Input**: Attach dataset chứa `memory_sft_code.zip` và dataset dữ liệu đã chuẩn bị (nếu có).


## 1. Cài đặt thư viện trực tiếp vào môi trường Kaggle
Kaggle đã cài sẵn PyTorch GPU + CUDA cho card 2×T4. Cài đặt trực tiếp các thư viện bổ trợ của LeRobot và SFT (không dùng venv để tránh lỗi ensurepip/xung đột NumPy):


In [1]:
!pip install --upgrade pip
!pip install "lerobot[smolvla,peft,dataset]>=0.4.0" "peft>=0.12.0" "accelerate>=0.34.0" "transformers>=4.48.0" "h5py>=3.10.0" "huggingface-hub>=0.25.0"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 141.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 110.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 120.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.8 MB/s  0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=6bd9232348a7c21f0620ca597992b32609404681ba4510647f91b9fb7d4b386f
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt
  Attempting uninstall: Pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0

## 2. Tải Checkpoint gốc và Tokenizer (Assets)
Tải checkpoint `lerobot/smolvla_libero` và tokenizer `SmolVLM2-500M-Video-Instruct` đã được pin commit và xác thực mã băm SHA256:


In [2]:
import shutil
from pathlib import Path

# Tìm thư mục code trong /kaggle/input và đồng bộ vào /kaggle/working
code_scripts = list(Path("/kaggle/input").rglob("scripts/prepare_memory_sft.py"))
if code_scripts:
    code_root = code_scripts[0].parents[1]
    print(f"Đang đồng bộ mã nguồn từ {code_root} sang /kaggle/working...")
    for item in code_root.iterdir():
        dest = Path("/kaggle/working") / item.name
        if not dest.exists():
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)
    print("Mã nguồn đã sẵn sàng trong /kaggle/working!")


Đang đồng bộ mã nguồn từ /kaggle/input/datasets/luquanglinh/memory-sft-code sang /kaggle/working...
Mã nguồn đã sẵn sàng trong /kaggle/working!


In [3]:
from pathlib import Path

# Đường dẫn tới dataset chứa code nguồn
REPO = Path("/kaggle/input/datasets/luquanglinh/memory-sft-code/")
ASSETS_DIR = Path("/kaggle/temp/sft-assets")

# Chạy script bằng đường dẫn tuyệt đối đến file trong /kaggle/input
!python {REPO}/scripts/memory_sft_assets.py fetch {ASSETS_DIR}


model.safetensors: downloading bytes: ████▋                 |  191MB, 14.3MB/s  
model.safetensors: reconstructing file:  22%|█▎    |  201MB /  907MB            
model.safetensors: reconstructing file:  25%|█▌    |  228MB /  907MB, 18.4MB/s  
model.safetensors: downloading bytes: ███████▊              |  321MB, 25.5MB/s  
model.safetensors: downloading bytes: ██████████████████▎   |  753MB, 61.0MB/s  
model.safetensors: reconstructing file:  44%|██▋   |  401MB /  907MB, 24.3MB/s  
model.safetensors: reconstructing file:  61%|███▋  |  557MB /  907MB, 30.1MB/s  
model.safetensors: downloading bytes: ██████████████████████|  778MB, 59.4MB/s  
model.safetensors: reconstructing file: 100%|██████|  907MB /  907MB, 74.0MB/s  
config.json: 100%|█████████████████████████| 2.42k/2.42k [00:00<00:00, 9.23MB/s]
policy_preprocessor.json: 100%|████████████| 2.04k/2.04k [00:00<00:00, 8.96MB/s]

policy_preprocessor_step_5_normalizer_pr(…): downloading bytes:  |  0.00B       
policy_preprocessor_step_5

## 3. Chọn và thiết lập Dữ liệu (Dataset)
- Khuyến nghị: Upload dataset đã chuẩn bị (có `manifest.json` và `shards/`) vào `/kaggle/input`.
- Hoặc chuẩn bị trực tiếp trên Kaggle: đặt `PREPARE_ON_KAGGLE = True`.


In [4]:
!rm -rf /kaggle/working/libero10-memory

In [5]:
import json
from pathlib import Path

PREPARE_ON_KAGGLE = True
DATA_DIR = Path("/kaggle/input/libero10-memory")
TASK_NAMES = ["KITCHEN_SCENE4_put_the_black_bowl_in_the_bottom_drawer_of_the_cabinet_and_close_it"]  # ví dụ: ["STUDY_SCENE1_pick_up_the_book_and_place_it_in_the_back_compartment_of_the_caddy"]

if PREPARE_ON_KAGGLE:
    DATA_DIR = Path("/kaggle/working/libero10-memory")
    extra = " ".join([f"--task-names {t}" for t in TASK_NAMES])
    !python /kaggle/input/datasets/luquanglinh/memory-sft-code/scripts/prepare_memory_sft.py --suite libero_10 --source-root /kaggle/temp/libero-raw --output-dir {DATA_DIR} --tokenizer-path {ASSETS_DIR}/backbone --download --source-lock resources/manifests/libero_10_sft_sources.json {extra}
else:
    # Tự động tìm manifest.json trong /kaggle/input nếu DATA_DIR mặc định chưa khớp
    if not (DATA_DIR / "manifest.json").is_file():
        manifests = list(Path("/kaggle/input").rglob("manifest.json"))
        if manifests:
            DATA_DIR = manifests[0].parent
            print(f"Auto-detected dataset tại: {DATA_DIR}")

if not (DATA_DIR / "manifest.json").is_file():
    raise FileNotFoundError("Không tìm thấy manifest.json! Hãy attach dataset đã chuẩn bị hoặc đặt PREPARE_ON_KAGGLE = True.")

manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print("Suite:", manifest.get("suite", "unknown"), "| Commit:", manifest.get("source_revision", "unknown"))
for row in manifest.get("sources", []):
    print("Task:", row.get("path"), "| Split frames:", row.get("frame_memory_counts_by_split"))


NOTE: this base-LIBERO converter uses the legacy proprioception writer. For the new T1/T3 RGB writer use prepare_libero_mem_sft.py; legacy memory is not validated training evidence.

libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes:  | 84.1MB, 5.71M
libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes:  |  109MB, 9.49M
libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes: ▏|  168MB, 13.7M
libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes: ▏|  190MB, 15.8M
libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes: ▍|  549MB, 44.1M
libero_10/KITCHEN_SCENE4_put_the_black_b(…): reconstructing file:  39%|▍|  482MB
libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes: ▍|  584MB, 49.3M
libero_10/KITCHEN_SCENE4_put_the_black_b(…): reconstructing file:  63%|▋|  775MB
libero_10/KITCHEN_SCENE4_put_the_black_b(…): reconstructing file:  86%|▊| 1.06GB
libero_10/KITCHEN_SCENE4_put_the_black_b(…): downloading bytes: █|  588MB, 49.8M
libero_

## 4. Smoke test trên 2× GPU T4 (2 optimizer updates)
Xác nhận huấn luyện song song Accelerate DDP phân bổ đúng 2 GPU, FP16 AMP không bị OOM trước khi chạy dài:


In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [7]:
path = "/kaggle/working/scripts/train_memory_sft.py"

with open(path, "r", encoding="utf-8") as f:
    content = f.read()

# Sửa dòng kiểm tra để chấp nhận cả action_is_pad, actions_is_pad và actions_id_pad
old_check = 'if "actions_id_pad" not in inspect.getsource(SmolVLAPolicy.forward):'
new_check = 'forward_src = inspect.getsource(SmolVLAPolicy.forward)\n    if "action_is_pad" not in forward_src and "actions_is_pad" not in forward_src and "actions_id_pad" not in forward_src:'

content = content.replace(old_check, new_check)

# Đảm bảo truyền cả hai khóa vào batch để thỏa mãn cả dataset lẫn LeRobot
old_batch = 'batch["actions_id_pad"] = padding.to(accelerator.device)'
new_batch = 'pad = padding.to(accelerator.device)\n                    batch["actions_id_pad"] = pad\n                    batch["action_is_pad"] = pad'

content = content.replace(old_batch, new_batch)

with open(path, "w", encoding="utf-8") as f:
    f.write(content)

# In kiểm tra các dòng 134-140 xem đã thay đổi chưa:
with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()
print("".join(lines[134:141]))


    trainable = {n: p.numel() for n, p in policy.named_parameters() if p.requires_grad}
    if not trainable or any("vlm_with_expert.vlm." in name for name in trainable):
        raise RuntimeError("Unexpected trainable parameter set: backbone must stay frozen")
    try:
        forward_src = inspect.getsource(SmolVLAPolicy.forward)
        has_pad = any(k in forward_src for k in ("action_is_pad", "actions_is_pad", "actions_id_pad"))
    except Exception:



In [8]:
SMOKE_OUTPUT = "/kaggle/working/sft-smoke"

!accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no \
  scripts/train_memory_sft.py \
  --assets-dir {ASSETS_DIR} \
  --data-dir {DATA_DIR} \
  --output-dir {SMOKE_OUTPUT} \
  --expected-gpus 2 \
  --steps 2 \
  --warmup-steps 0 \
  --validation-max-batches 2 \
  --save-every 1


Reducing the number of VLM layers to 16 ...
Reducing the number of VLM layers to 16 ...
Loading weights from local directory
Loading weights from local directory
[rank0]:[W1026 06:53:38.612881747 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
[rank1]:[W1026 06:53:38.613364591 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adve

## 5. Huấn luyện chính thức (Training)
Preset: frozen VLM backbone, expert LoRA rank 16 / alpha 16, FP16, batch 2/GPU x accumulation 8 = effective batch 32.


In [9]:
RUN_TRAINING = True  # Đổi thành True khi smoke test ở bước 4 đã qua
CONDITION = "memory"   # 'memory' hoặc 'no_memory' (đối chứng)
OUTPUT_DIR = f"/kaggle/working/sft-{CONDITION}"

if RUN_TRAINING:
    !accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no \
      scripts/train_memory_sft.py \
      --assets-dir {ASSETS_DIR} \
      --data-dir {DATA_DIR} \
      --output-dir {OUTPUT_DIR} \
      --expected-gpus 2 \
      --condition {CONDITION} \
      --adaptation expert_lora \
      --steps 3000


Reducing the number of VLM layers to 16 ...
Reducing the number of VLM layers to 16 ...
Loading weights from local directory
Loading weights from local directory
[rank0]:[W1026 06:54:57.697172174 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
[rank1]:[W1026 06:54:57.704228221 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adve

## 6. Xuất và tải Checkpoint về máy Local
Mô hình đã merge LoRA sẽ nằm tại: `{OUTPUT_DIR}/pretrained_model/`.
Tải thư mục này về máy local và kiểm định bằng:
```bash
python scripts/validate_memory_sft_export.py --checkpoint /path/to/pretrained_model --data-dir experiments/pilot_sft_test
```
